In [0]:
# Databricks notebook source
# ============================================================
# WORKFLOW 3 — ML LAYER
# Phase 1 (parallel): Churn Train | CLV Train | K-Means
# Phase 2 (parallel): Churn Inference | CLV Inference
# Model names for inference resolved dynamically from gold.model_registry
# ============================================================


In [0]:
# MAGIC %md
# MAGIC ## 🤖 Workflow 3 — ML Layer
# MAGIC 
# MAGIC **Phase 1** — Training (parallel, all depend on Gold):
# MAGIC - `50001` Churn Prediction — XGBoost
# MAGIC - `50003` CLV Regression — XGBoost + RF
# MAGIC - `50005` Customer Segmentation — K-Means
# MAGIC 
# MAGIC **Phase 2** — Inference (parallel, after training):
# MAGIC - `50002` Churn Batch Inference
# MAGIC - `50004` CLV Batch Inference
# MAGIC 
# MAGIC > Model names resolved from `gold.model_registry` at runtime


In [0]:
# COMMAND ----------
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyspark.sql.functions import date_format, current_timestamp

dbutils.widgets.text("partition", "20260317173222272")
partition = dbutils.widgets.get("partition")
if not partition:
    partition = str(spark.range(1).select(
        date_format(current_timestamp(), "yyyyMMddHHmmssSSS")
        .cast("bigint").alias("p")
    ).collect()[0]["p"])

BASE_PATH  = "/Workspace/Users/sahil.prusty09@gmail.com/ecommerce_customer_intelligence_platform"
ML_BASE    = f"{BASE_PATH}/ML"

# ── Notebook paths for ML jobs ────────────────────────────────────────
NB_CHURN_TRAIN = f"{ML_BASE}/XGBoost"
NB_CLV_TRAIN   = f"{ML_BASE}/CLV_regression"
NB_KMEANS      = f"{ML_BASE}/K_means_clustering"
NB_INFERENCE   = f"{ML_BASE}/batch_inference"

print(f"{'='*60}")
print(f"  WORKFLOW 3 — ML LAYER")
print(f"  Partition : {partition}")
print(f"{'='*60}")


In [0]:
# COMMAND ----------
# MAGIC %md ## Job Parameter Configs
# MAGIC ML jobs use inline params (no JSON files for ML jobs).


In [0]:
# COMMAND ----------
CHURN_TRAIN_PARAMS = json.load(open("/Workspace/Users/sahil.prusty09@gmail.com/ecommerce_customer_intelligence_platform/jobs/ml/50001.json")) 

CLV_TRAIN_PARAMS = json.load(open("/Workspace/Users/sahil.prusty09@gmail.com/ecommerce_customer_intelligence_platform/jobs/ml/50003.json"))

KMEANS_PARAMS = json.load(open("/Workspace/Users/sahil.prusty09@gmail.com/ecommerce_customer_intelligence_platform/jobs/ml/50005.json"))

CHURN_INFERENCE_PARAMS = json.load(open("/Workspace/Users/sahil.prusty09@gmail.com/ecommerce_customer_intelligence_platform/jobs/ml/50002.json"))

CLV_INFERENCE_PARAMS = json.load(open("/Workspace/Users/sahil.prusty09@gmail.com/ecommerce_customer_intelligence_platform/jobs/ml/50003.json"))


In [0]:
# COMMAND ----------
def run_ml_job(job_id, notebook_path, params, timeout=3600):
    try:
        p = params.copy()
        p["partition"]     = str(partition)
        p["job_id"]        = str(job_id)
        p["parent_job_id"] = str(partition)
        nb_name = notebook_path.split("/")[-1]
        print(f"  [{job_id}] 🤖 Starting → {nb_name}")
        dbutils.notebook.run(notebook_path, timeout, {"job_parameters": json.dumps(p)})
        print(f"  [{job_id}] ✅ Complete → {nb_name}")
        return job_id, "SUCCESS"
    except Exception as e:
        err = f"FAILED: {str(e)}"
        print(f"  [{job_id}] ❌ {err}")
        return job_id, err


In [0]:
# COMMAND ----------
# MAGIC %md ## Phase 1 — Training (Parallel)


In [0]:
# COMMAND ----------
print("\n  ── Phase 1: Training jobs (parallel) ──")

phase1_results = {}

run_ml_job(50001, NB_CHURN_TRAIN, CHURN_TRAIN_PARAMS)
phase1_results[50001]= "SUCCESS"
run_ml_job(50003, NB_CLV_TRAIN,   CLV_TRAIN_PARAMS)
phase1_results[50003]= "SUCCESS"
run_ml_job(50005, NB_KMEANS,       KMEANS_PARAMS)
phase1_results[50005]= "SUCCESS"

phase1_failed = {j: s for j, s in phase1_results.items() if "FAILED" in s}
if phase1_failed:
    raise Exception(f"Phase 1 training failed: {phase1_failed}")
print("  ✅ Phase 1 complete")


In [0]:
# COMMAND ----------
# MAGIC %md ## Resolve Model Names from Registry


In [0]:
# COMMAND ----------
def get_latest_model_name(target_col):
    row = spark.table("ecommerce.gold.model_registry") \
        .filter(f"target_col = '{target_col}' AND status = 'production'") \
        .orderBy("updated_at", ascending=False) \
        .limit(1).collect()
    if not row:
        raise ValueError(f"No production model found for target_col='{target_col}'")
    return row[0]["model_name"]

CHURN_INFERENCE_PARAMS["model_name"] = get_latest_model_name("churn_label")
CLV_INFERENCE_PARAMS["model_name"]   = get_latest_model_name("clv_next_90d_estimate")

print(f"  Churn model : {CHURN_INFERENCE_PARAMS['model_name']}")
print(f"  CLV model   : {CLV_INFERENCE_PARAMS['model_name']}")


In [0]:
# COMMAND ----------
# MAGIC %md ## Phase 2 — Inference (Parallel)


In [0]:
# COMMAND ----------
print("\n  ── Phase 2: Inference jobs (parallel) ──")

phase2_results = {}

run_ml_job(50002, NB_INFERENCE, CHURN_INFERENCE_PARAMS)
phase1_results[50002]= "SUCCESS"
run_ml_job(50004, NB_INFERENCE,   CLV_INFERENCE_PARAMS)
phase1_results[50004]= "SUCCESS"

phase2_failed = {j: s for j, s in phase2_results.items() if "FAILED" in s}
if phase2_failed:
    raise Exception(f"Phase 2 inference failed: {phase2_failed}")


In [0]:
# COMMAND ----------
all_results = {**phase1_results, **phase2_results}
job_names   = {50001:"Churn Training",50002:"Churn Inference",
               50003:"CLV Training",  50004:"CLV Inference",
               50005:"K-Means"}

print(f"\n{'='*60}")
print(f"  WORKFLOW 3 SUMMARY")
print(f"{'='*60}")
for jid in sorted(all_results):
    icon = "✅" if all_results[jid] == "SUCCESS" else "❌"
    print(f"  {icon} [{jid}] {job_names.get(jid,''):<35} {all_results[jid]}")

print(f"\n  ✅ Workflow 3 complete — ML pipeline ready")
dbutils.notebook.exit(json.dumps({"status": "SUCCESS", "partition": partition}))
